In [1]:
# Bring in the spatial handler and math library
import geopandas as gpd
import numpy as np
# Bring in the tool to draw physical box shapes
from shapely.geometry import box

# Load your clean, clipped building package
buildings = gpd.read_file('data/Islamabad_Buildings_Clean.gpkg')

# Get the absolute outer limits of the city (Left, Bottom, Right, Top)
minx, miny, maxx, maxy = buildings.total_bounds

# Define the grid cell size (0.005 decimal degrees is roughly 500m x 500m)
cell_size = 0.005

# Create the grid cells by looping through the coordinates left-to-right, bottom-to-top
grid_cells = []
for x0 in np.arange(minx, maxx, cell_size):
    for y0 in np.arange(miny, maxy, cell_size):
        # Calculate the opposite corner of the current square
        x1 = x0 + cell_size
        y1 = y0 + cell_size
        # Draw the square and add it to our list
        grid_cells.append(box(x0, y0, x1, y1))

# Convert the raw squares into a spatial table matching the building coordinates
grid = gpd.GeoDataFrame(grid_cells, columns=['geometry'], crs=buildings.crs)

# Count how many buildings fall inside each grid cell based on physical intersection
buildings_in_grid = gpd.sjoin(buildings, grid, how='inner', predicate='intersects')

# Group the results by the grid cell ID and count them
density = buildings_in_grid.groupby('index_right').size().reset_index(name='building_count')

# Attach the counts back to the actual grid shapes
grid = grid.merge(density, left_index=True, right_on='index_right', how='left')

# Fill any empty cells with zero so the math doesn't break later
grid['building_count'] = grid['building_count'].fillna(0)

# Save the density grid as a new GeoPackage
grid.to_file('data/Islamabad_Density_Grid.gpkg', driver='GPKG')

# Print a success message showing the highest density found
max_count = int(grid['building_count'].max())
print(f"Grid created. The densest 500m block has {max_count} buildings.")

Grid created. The densest 500m block has 2311 buildings.
